In [0]:
# ==========================================
# GOLD LAYER NOTEBOOK
# ==========================================

from pyspark.sql import functions as F

# ------------------------------------------
# 1. Create Gold Schema (Run Once)
# ------------------------------------------

spark.sql("CREATE SCHEMA IF NOT EXISTS retail_catalog1.gold")

# ------------------------------------------
# 2. Read Silver Table
# ------------------------------------------

silver_df = spark.read.table(
    "retail_catalog1.silver.transactions"
)

# ------------------------------------------
# 3. Create Business KPIs
# ------------------------------------------

gold_df = (
    silver_df
    .groupBy("store_id")
    .agg(
        F.count("transaction_id").alias("total_transactions"),
        F.sum("amount").alias("total_sales"),
        F.avg("amount").alias("avg_transaction_value"),
        F.max("amount").alias("max_transaction"),
        F.min("amount").alias("min_transaction")
    )
)

# ------------------------------------------
# 4. Save Gold Table
# ------------------------------------------

(
    gold_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("retail_catalog1.gold.store_sales_summary")
)

# ------------------------------------------
# 5. Create Additional Payment Analytics
# ------------------------------------------

payment_summary = (
    silver_df
    .groupBy("payment_type")
    .agg(
        F.count("*").alias("transaction_count"),
        F.sum("amount").alias("total_amount")
    )
)

(
    payment_summary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("retail_catalog1.gold.payment_summary")
)

# ------------------------------------------
# 6. Create Daily Sales Summary
# ------------------------------------------

daily_sales = (
    silver_df
    .withColumn(
        "sale_date",
        F.to_date(F.col("timestamp").cast("string"))
    )
    .filter(F.col("sale_date").isNotNull())
    .groupBy("sale_date")
    .agg(
        F.sum("amount").alias("daily_sales"),
        F.count("*").alias("daily_transactions")
    )
)


(
    daily_sales.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("retail_catalog1.gold.daily_sales_summary")
)

# ------------------------------------------
# 7. Display Results
# ------------------------------------------

print("===== STORE SALES SUMMARY =====")
display(
    spark.table("retail_catalog1.gold.store_sales_summary")
)

print("===== PAYMENT SUMMARY =====")
display(
    spark.table("retail_catalog1.gold.payment_summary")
)

print("===== DAILY SALES SUMMARY =====")
display(
    spark.table("retail_catalog1.gold.daily_sales_summary")
)

print("✅ Gold Layer Completed Successfully")

===== STORE SALES SUMMARY =====


store_id,total_transactions,total_sales,avg_transaction_value,max_transaction,min_transaction
S02,12,11041.0,920.0833333333334,2500.0,350.0
S01,15,20071.5,1338.1,3500.0,420.0
S03,14,359400.0,25671.428571428572,100000.0,300.0
S04,8,3601.5,450.1875,800.0,150.0


===== PAYMENT SUMMARY =====


payment_type,transaction_count,total_amount
Credit,20,376801.0
UPI,10,5601.5
Cash,11,4931.0
Debit,8,6780.5


===== DAILY SALES SUMMARY =====


sale_date,daily_sales,daily_transactions
2025-06-01,384884.0,36
2025-06-02,8480.0,12


✅ Gold Layer Completed Successfully
